# Demo 8 — Explanation-Guided Attacks
## Saliency-Targeted Adversarial Examples using Captum

**Core Idea:** Instead of perturbing all pixels uniformly (like FGSM),
first **explain** the model (identify which pixels matter most),
then **target only those pixels** with perturbations.

**Result:** More efficient, more transferable, more stealthy attacks.

**Library:** `captum` (PyTorch Model Interpretability)

In [ ]:
# Install
# !pip install torch torchvision captum matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from captum.attr import Saliency, IntegratedGradients, GuidedBackprop
torch.manual_seed(42)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
train_set_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set_f  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set = torch.utils.data.Subset(train_set_f, range(1000))
test_set  = torch.utils.data.Subset(test_set_f, range(200))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)
print('Captum + data ready.')

## Step 1 — Train the Victim Model

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features   = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.classifier(self.features(x))

model = CNN().to(DEVICE)
opt   = optim.Adam(model.parameters(), lr=1e-3)
fn    = nn.CrossEntropyLoss()
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
for epoch in range(3):
    model.train()
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(model(X), y).backward(); opt.step()
model.eval()
correct = sum((model(X.to(DEVICE)).argmax(1)==y.to(DEVICE)).sum().item()
               for X, y in test_loader)
print(f'Model accuracy: {correct/len(test_set)*100:.2f}%')

## Step 2 — Generate Explanation Maps

| Method | Description |
|--------|-------------|
| **Saliency** | Gradient magnitude w.r.t. input pixels |
| **Integrated Gradients** | Average gradient along path from baseline to input |
| **Guided Backprop** | Backprop with positive-only rule for sharper maps |

In [ ]:
saliency_attr = Saliency(model)
ig_attr       = IntegratedGradients(model)
guided_attr   = GuidedBackprop(model)

X_sample, y_sample = next(iter(torch.utils.data.DataLoader(test_set, batch_size=6)))
X_sample = X_sample.to(DEVICE).requires_grad_(True)
y_sample = y_sample.to(DEVICE)
preds    = model(X_sample).argmax(1)

sal_maps = saliency_attr.attribute(X_sample, target=preds).abs()
ig_maps  = ig_attr.attribute(X_sample, target=preds, n_steps=50).abs()
gb_maps  = guided_attr.attribute(X_sample, target=preds).abs()
print('Attribution maps computed.')

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(16, 10))
fig.suptitle('Explanation Maps: What Does the Model Look At?', fontsize=14, fontweight='bold')
for i in range(6):
    imgs_list = [
        X_sample[i].squeeze().cpu().detach().numpy(),
        sal_maps[i].squeeze().cpu().detach().numpy(),
        ig_maps[i].squeeze().cpu().detach().numpy(),
        gb_maps[i].squeeze().cpu().detach().numpy()
    ]
    cmaps = ['gray', 'hot', 'hot', 'hot']
    titles= [f'Input\nTrue:{y_sample[i].item()} Pred:{preds[i].item()}',
             'Saliency', 'Integr. Grad.', 'Guided BP']
    for row, (img, cmap, title) in enumerate(zip(imgs_list, cmaps, titles)):
        axes[row, i].imshow(img, cmap=cmap)
        axes[row, i].set_title(title, fontsize=8)
        axes[row, i].axis('off')
plt.tight_layout()
plt.savefig('08a_explanation_maps.png', dpi=150)
plt.show()

## Step 3 — Explanation-Guided Attack

**Key idea:** Compute a saliency mask → keep only the top-K% most important
pixels → apply FGSM perturbation **only to those pixels**.

Compare against standard FGSM (all pixels) to show fewer pixels are needed.

In [ ]:
def fgsm(model, X, y, epsilon):
    Xa = X.clone().detach().requires_grad_(True)
    fn(model(Xa), y).backward()
    return (X + epsilon * Xa.grad.sign()).clamp(0, 1).detach()

def explanation_guided_attack(model, X, y, epsilon, top_k_pct=0.15):
    Xa = X.clone().detach().requires_grad_(True)
    fn(model(Xa), y).backward()
    grad_sign = Xa.grad.sign().detach()
    sal       = Xa.grad.abs().detach()
    threshold = torch.quantile(sal.view(sal.shape[0], -1), 1-top_k_pct, dim=1)
    mask      = (sal >= threshold.view(-1,1,1,1)).float()
    return (X + epsilon * grad_sign * mask).clamp(0, 1).detach()

epsilon    = 0.2
top_k_vals = [0.05, 0.10, 0.15, 0.25, 0.50, 1.00]
results    = {}
model.eval()
for k in top_k_vals:
    correct = 0; total = 0
    for X, y in test_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        X_adv = fgsm(model, X, y, epsilon) if k == 1.00 else explanation_guided_attack(model, X, y, epsilon, k)
        correct += (model(X_adv).argmax(1) == y).sum().item()
        total   += y.size(0)
    results[k] = correct / total

print(f'{'Top-K %':>8}  {'Accuracy After Attack':>22}')
print('-' * 35)
for k, acc in results.items():
    print(f'{k*100:>7.1f}%  {acc*100:>21.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ks   = [k*100 for k in top_k_vals]
accs = [results[k]*100 for k in top_k_vals]
ax.plot(ks, accs, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax.set_xlabel('% of Pixels Perturbed')
ax.set_ylabel('Model Accuracy (%) — lower = stronger attack')
ax.set_title('Explanation-Guided Attack Efficiency\n(rightmost = standard FGSM all-pixel)', fontweight='bold')
ax.annotate('Standard FGSM\n(all pixels)', xy=(100, accs[-1]),
             xytext=(70, accs[-1]+5), arrowprops=dict(arrowstyle='->'))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('08b_explanation_attack.png', dpi=150)
plt.show()

## Summary

Explanation-guided attacks show **model interpretability tools can be weaponised**.
By targeting only important pixels:
- The attack is **more stealthy** (fewer pixels changed)
- Perturbations are **semantically meaningful**
- Transfer attacks are **more robust** across architectures

**Defenses:** Hiding explanation maps (hard labels only), randomising the explanation
method, adversarial training on explanation-targeted examples.